# Task 7: Low-Rank Adaptation (LoRA) Matrix Projection Fine-Tuning of a 3B Model

## Formula
$$W^\prime = W_0 + \frac{\alpha}{r} (B \cdot A)$$


In [1]:
import torch
import torch.nn as nn

# Custom LoRA layer attaching rank-decomposed matrices A and B parallel to frozen linear projections
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=4, lora_alpha=8):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad = False # Freeze base model parameter
        
        self.r = r
        self.scaling = lora_alpha / r
        self.lora_A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

    def forward(self, x):
        return self.linear(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scaling


In [2]:
# Instantiate 4096-dim LoRA projection layer and run simulated domain adaptation training step
layer = LoRALinear(in_features=4096, out_features=4096, r=4)

base_params = sum(p.numel() for p in layer.parameters() if not p.requires_grad)
lora_params = sum(p.numel() for p in layer.parameters() if p.requires_grad)

optimizer = torch.optim.AdamW([layer.lora_A, layer.lora_B], lr=1e-3)
x = torch.randn(2, 4096)
target = torch.randn(2, 4096)

# 3 Training steps
losses = []
for _ in range(3):
    optimizer.zero_grad()
    out = layer(x)
    loss = torch.mean((out - target)**2)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Base Layer Parameters (Frozen): {base_params:,}")
print(f"LoRA Adapter Parameters (Trainable): {lora_params:,}")
print(f"Trainable Footprint Ratio: {100 * lora_params / (base_params + lora_params):.2f}%")
print("LoRA Fine-tuning Loss Trajectory:", [round(l, 4) for l in losses])


Base Layer Parameters (Frozen): 16,777,216
LoRA Adapter Parameters (Trainable): 36,864
Trainable Footprint Ratio: 0.22%
LoRA Fine-tuning Loss Trajectory: [1.3, 1.295, 1.2555]
